In [1]:
!pip -q install transformers==4.52.4
!pip install huggingface-hub==0.36.2
!pip -q install datasets
!pip -q install sentencepiece
!pip -q install sacrebleu
!pip -q install bert-score
!pip -q install evaluate
!pip -q install accelerate

In [2]:
import os
import random
import time
import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

print(torch.cuda.get_device_name(0))

Tesla T4


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Datasets.zip to Datasets (2).zip


In [4]:
import zipfile

zipfile.ZipFile("Datasets.zip").extractall("dataset")

In [5]:
import os

for root, dirs, files in os.walk("dataset"):
    for f in files:
        print(f)

train_en_10000.csv
test_en_1000.csv
train_sa_10000.csv
dev_sa_1000.csv
dev_en_1000.csv
test_sa_1000.csv


In [6]:
train_sa = pd.read_csv("dataset/train_sa_10000.csv")
train_en = pd.read_csv("dataset/train_en_10000.csv")

dev_sa = pd.read_csv("dataset/dev_sa_1000.csv")
dev_en = pd.read_csv("dataset/dev_en_1000.csv")

test_sa = pd.read_csv("dataset/test_sa_1000.csv")
test_en = pd.read_csv("dataset/test_en_1000.csv")

In [7]:
print("train_sa columns:", train_sa.columns.tolist())
print("train_en columns:", train_en.columns.tolist())

print("\nFirst Sanskrit rows")
print(train_sa.head())

print("\nFirst English rows")
print(train_en.head())

train_sa columns: ['Source_id', 'Sentence_sa']
train_en columns: ['Source_id', 'Sentence_en']

First Sanskrit rows
   Source_id                                        Sentence_sa
0          1                         "Ctrl, S नुत्वा रक्षन्तु।"
1          2                     गुरुः छात्रान् एकवारं पाठयति ।
2          3  चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...
3          4       वयं  Colors विकल्पं तस्योपरि नोदनेन चिनुमः ।
4          5  "अत्र कानिचन उदाहरणानि पश्याम:- एक: पर्वत:, चत...

First English rows
   Source_id                                        Sentence_en
0          1                              Save it with Ctrl, S.
1          2         Teacher will teach the students only once.
2          3  To recreate this animation, I have to take two...
3          4    I will choose Colors options by clicking on it.
4          5  "See the example here - one mountain, four vil...


In [8]:
train = train_sa.merge(train_en,on="Source_id")
dev = dev_sa.merge(dev_en,on="Source_id")
test = test_sa.merge(test_en,on="Source_id")

train.head()

,Source_id,Sentence_sa,Sentence_en
0,1,"""Ctrl, S नुत्वा रक्षन्तु।""","Save it with Ctrl, S."
1,2,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,3,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"To recreate this animation, I have to take two..."
3,4,वयं Colors विकल्पं तस्योपरि नोदनेन चिनुमः ।,I will choose Colors options by clicking on it.
4,5,"""अत्र कानिचन उदाहरणानि पश्याम:- एक: पर्वत:, चत...","""See the example here - one mountain, four vil..."


In [9]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train)
dev_ds = Dataset.from_pandas(dev)
test_ds = Dataset.from_pandas(test)

In [10]:
from transformers import AutoTokenizer

MODEL="facebook/nllb-200-distilled-600M"

tokenizer=AutoTokenizer.from_pretrained(MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [11]:
SRC_LANG="san_Deva"
TGT_LANG="eng_Latn"

tokenizer.src_lang=SRC_LANG
tokenizer.tgt_lang=TGT_LANG

In [12]:
MAX_LEN = 128

def preprocess(batch):

    model_inputs = tokenizer(
        batch["Sentence_sa"],
        max_length=MAX_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["Sentence_en"],
        max_length=MAX_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [13]:
train_ds = train_ds.map(preprocess, batched=True)
dev_ds = dev_ds.map(preprocess, batched=True)
test_ds = test_ds.map(preprocess, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [14]:
from transformers import AutoModelForSeq2SeqLM

model=AutoModelForSeq2SeqLM.from_pretrained(MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100SdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
   

In [15]:
num_params=sum(p.numel() for p in model.parameters())

print("Parameters:",num_params)

Parameters: 615073792


In [16]:
from transformers import DataCollatorForSeq2Seq

collator=DataCollatorForSeq2Seq(tokenizer,model=model)

In [17]:
import transformers
print(transformers.__version__)


4.52.4


In [18]:
from transformers import Seq2SeqTrainingArguments
import inspect

print(inspect.signature(Seq2SeqTrainingArguments.__init__))

(self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Union[dict, str, NoneType] = <factory>, warmup_ratio: float = 0.0, warmup_ste

In [19]:
from transformers import Seq2SeqTrainingArguments
import torch

args = Seq2SeqTrainingArguments(

    output_dir="checkpoint",

    learning_rate=3e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,


    gradient_accumulation_steps=4,

    weight_decay=0.01,

    num_train_epochs=5,

    predict_with_generate=True,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    save_total_limit=1,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=42
)

In [20]:
import evaluate

bleu=evaluate.load("sacrebleu")

In [21]:
import numpy as np

def compute_metrics(eval_pred):

    predictions,labels=eval_pred

    decoded_preds=tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels=np.where(labels!=-100,labels,tokenizer.pad_token_id)

    decoded_labels=tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result=bleu.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    return {"BLEU":result["score"]}

In [22]:
from transformers import Seq2SeqTrainer
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(

    model=model,

    args=args,

    train_dataset=train_ds,

    eval_dataset=dev_ds,

    tokenizer=tokenizer,

    data_collator=collator,

    compute_metrics=compute_metrics,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/tmp/ipykernel_14368/2762269934.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [23]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [24]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu
1,1.615200,1.485589,28.001372
2,1.415500,1.421051,29.382168
3,1.264200,1.392465,30.540650
4,1.202100,1.385666,30.301274
5,1.158800,1.386653,30.692102


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3125, training_loss=1.3460783782958985, metrics={'train_runtime': 4909.8028, 'train_samples_per_second': 10.184, 'train_steps_per_second': 0.636, 'total_flos': 4453560797134848.0, 'train_loss': 1.3460783782958985, 'epoch': 5.0})

In [32]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")

Total Parameters      : 615,073,792
Trainable Parameters  : 615,073,792


In [25]:
from bert_score import score

preds = []

start = time.time()

model.eval()

for sent in test["Sentence_sa"]:

    inputs = tokenizer(
        sent,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = model.generate(

        **inputs,

        forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),

        num_beams=5,

        length_penalty=1.2,

        no_repeat_ngram_size=3,

        max_length=128
    )

    preds.append(
        tokenizer.decode(
            output[0],
            skip_special_tokens=True
        )
    )

end = time.time()

print("Inference Time:", end-start)

Inference Time: 584.2857384681702


In [26]:
P,R,F=score(
    preds,
    test["Sentence_en"].tolist(),
    lang="en",
    rescale_with_baseline=True
)

print(F.mean())

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor(0.5982)


In [27]:
import sacrebleu

bleu=sacrebleu.corpus_bleu(
    preds,
    [test["Sentence_en"].tolist()]
)

print(bleu.score)

29.85486628578593


In [28]:
submission=pd.DataFrame()

submission["Source_id"]=test["Source_id"]

submission["Sentence_en"]=preds

submission.to_csv(
    "submission.csv",
    index=False,
    encoding="utf-8"
)

submission.head()

,Source_id,Sentence_en
0,1,It also helps in troubleshooting Eclipse program.
1,2,"""By faith I speak: even as it is written, We a..."
2,3,Then it will automatically search for the driv...
3,4,"For all iterations, the iterator is set to the..."
4,5,"""And when he had opened the second seal, I hea..."


In [29]:
from google.colab import files

files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
print(len(preds))
print(len(test["Sentence_en"]))

for i in range(5):
    print("="*80)
    print("Sanskrit :", test["Sentence_sa"].iloc[i])
    print("Reference:", test["Sentence_en"].iloc[i])
    print("Prediction:", preds[i])

1000
1000
Sanskrit : एक्लिप्स् इति प्रोग्रामर् कृते दोषान्वेषणे अपि साहाय्यं करोति।
Reference: Eclipse also helps the programmer to find out errors.
Prediction: It also helps in troubleshooting Eclipse program.
Sanskrit : विश्वासकारणादेव समभाषि मया वचः। इति यथा शास्त्रे लिखितं तथैवास्माभिरपि विश्वासजनकम् आत्मानं प्राप्य विश्वासः क्रियते तस्माच्च वचांसि भाष्यन्ते।
Reference: "We having the same spirit of faith, according as it is written, I believed, and therefore have I spoken; we also believe, and therefore speak;"
Prediction: "By faith I speak: even as it is written, We also have the spirit of faith,"
Sanskrit : तदा, तत्स्वयं ड्रैवर निमित्तम् अन्वेष्यति। अहं 'Cancel' इत्यस्योपरि नुदामि।
Reference: Then it will automatically begin searching for drivers. I will click on Cancel.
Prediction: Then it will automatically search for the driver. I will click on Cancel.
Sanskrit : सर्वेभ्यः इटरेशन्-अर्थम्, iterator इतीदं प्रत्येकस्मै इण्डेक्स्-वेल्यू-इत्यस्मै सेट् क्रियते । 1,1 पश्चात् 1,2  एव

In [31]:
from bert_score import score

P, R, F = score(
    preds,
    test["Sentence_en"].tolist(),
    model_type="roberta-large",
    num_layers=17,
    lang="en",
    rescale_with_baseline=False
)

print("Precision :", P.mean().item())
print("Recall    :", R.mean().item())
print("F1        :", F.mean().item())

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Precision : 0.9354918599128723
Recall    : 0.9291130304336548
F1        : 0.9321918487548828
